<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Misc/IPO_File_Creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loading and Merging IPO Data

I will load the two IPO files into pandas DataFrames. Then, I will merge them based on the common `Symbol` column and remove any duplicate columns.

In [29]:
import pandas as pd
import csv

# Define file paths
overview_file_path = '/content/IPO_2024_Overview.txt'
detail_file_path = '/content/IPO_2024_Detail.txt'
output_file_path = 'IPO_2024.csv' # Placeholder for output file

# Load the IPO Overview file
try:
    df_overview = pd.read_csv(overview_file_path, sep='|', quotechar='"')
    # Clean up column names by stripping whitespace and removing quotes
    df_overview.columns = df_overview.columns.str.strip().str.replace('"', '')
    print("IPO Overview DataFrame loaded successfully.")
    display(df_overview.head())
except Exception as e:
    print(f"Error loading {overview_file_path}: {e}")
    df_overview = pd.DataFrame() # Create an empty DataFrame to avoid errors later

IPO Overview DataFrame loaded successfully.


,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$1.20,-71.00%
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",-,$0.435,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$1.03,-73.50%
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.70,7.00%
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.67,6.70%


In [30]:
# Load the IPO Detail file
try:
    df_detail = pd.read_csv(detail_file_path, sep='|', quotechar='"')
    # Clean up column names by stripping whitespace and removing quotes
    df_detail.columns = df_detail.columns.str.strip().str.replace('"', '')
    print("IPO Detail DataFrame loaded successfully.")
    display(df_detail.head())
except Exception as e:
    print(f"Error loading {detail_file_path}: {e}")
    df_detail = pd.DataFrame() # Create an empty DataFrame to avoid errors later

IPO Detail DataFrame loaded successfully.


,Symbol,Exchange,IPO Price,Deal Size,Shares Offered
0,ONEG,NASDAQ,$4.00,7.00M,"1,750,000"
1,BYAH,NASDAQ,-,-,-
2,HIT,NASDAQ,$4.00,9.20M,"2,300,000"
3,TDAC,NASDAQ,$10.00,150.00M,"15,000,000"
4,RANG,NASDAQ,$10.00,100.00M,"10,000,000"


In [31]:
# Perform the merge on the 'Symbol' column
# Using an outer merge to include all symbols from both files.
# Suffixes are added to identify columns that exist in both DataFrames but are not the merge key.
merged_df = pd.merge(df_overview, df_detail, on='Symbol', how='inner', suffixes=('_overview', '_detail'))

# Handle redundant columns
# Iterate through columns and remove '_detail' versions, renaming '_overview' versions
for col in merged_df.columns:
    if col.endswith('_overview'):
        original_col_name = col.replace('_overview', '')
        if original_col_name + '_detail' in merged_df.columns:
            # If both _overview and _detail exist, keep _overview and drop _detail
            merged_df[original_col_name] = merged_df[col]
            merged_df = merged_df.drop(columns=[col, original_col_name + '_detail'])
        else:
            # If only _overview exists, just rename it
            merged_df = merged_df.rename(columns={col: original_col_name})
    elif col.endswith('_detail') and col.replace('_detail', '') not in merged_df.columns:
        # If only _detail exists for an original column, rename it
        merged_df = merged_df.rename(columns={col: col.replace('_detail', '')})

print("Merged DataFrame without redundant columns:")
display(merged_df.head())

Merged DataFrame without redundant columns:


,IPO Date,Symbol,Company Name,Current,Return,Exchange,Deal Size,Shares Offered,IPO Price
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$1.20,-71.00%,NASDAQ,7.00M,"1,750,000",$4.00
1,"Dec 27, 2024",BYAH,"Park Ha Biological Technology Co., Ltd.",$0.435,-,NASDAQ,-,-,-
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$1.03,-73.50%,NASDAQ,9.20M,"2,300,000",$4.00
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.70,7.00%,NASDAQ,150.00M,"15,000,000",$10.00
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.67,6.70%,NASDAQ,100.00M,"10,000,000",$10.00


In [32]:


# Save the merged DataFrame to a CSV file, ensuring all fields are quoted
merged_df.to_csv(output_file_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Merged data saved to {output_file_path}")

Merged data saved to IPO_2024.csv
